In [3]:
def get_com(x, y, z, m):
    m_sum = np.sum(m)
    return np.array([np.sum(x * m), np.sum(y * m), np.sum(z * m)]) / m_sum

# maybe mention profilling this in report

def shrinking_sphere(x, y, z, m, r=25, rate=0.8, history=None, limit=1000):
    if rate >= 1: return

    count = 0

    sq_r = r ** 2
    sq_rate = rate ** 2
    
    guess = np.array([np.median(axis)for axis in [x, y, z]])

    bound_x = x
    bound_y = y
    bound_z = z
    bound_m = m

    mask = np.ones(len(x), dtype=bool)
    
    while True:
        if history is not None:
            history.append(guess)
        
        if count > limit: 
            print('limit exceeded')
            return guess
        
        sq_radii = (bound_x - guess[0])**2 + (bound_y - guess[1])**2 + (bound_z - guess[2])**2
        
        idx = sq_radii <= sq_r

        if np.sum(idx) < 0.01 * len(x): return guess

        bound_x = bound_x[idx]
        bound_y = bound_y[idx]
        bound_z = bound_z[idx]
        bound_m = bound_m[idx]

        guess = get_com(bound_x, bound_y, bound_z, bound_m)

        sq_r *= sq_rate
        count += 1
        

    
    

In [1]:
def shrinking_sphere_old(x, y, z, m, r=50, rate=0.95, history=None, limit=1000):
    if rate >= 1: return

    count = 0
    
    guess = [np.median(axis)for axis in [x, y, z]]

    bound_x = x.copy()
    bound_y = y.copy()
    bound_z = z.copy()
    bound_m = m.copy()
    
    while True:
        if count > limit: 
            print('limit exceeded')
            return 
        
        if history is not None:
            history.append(guess)
        
        radii = np.sqrt( (bound_x - guess[0])**2 + (bound_y - guess[1])**2 + (bound_z - guess[2])**2 )
        
        idx = radii <= r

        if np.sum(idx) < 0.01 * len(x): return guess

        bound_x = bound_x[idx]
        bound_y = bound_y[idx]
        bound_z = bound_z[idx]
        bound_m = bound_m[idx]

        guess = get_com(bound_x, bound_y, bound_z, bound_m)

        r *= rate
        count += 1
        

    
    

In [ ]:
def find_mass_profile_old(x, y, z, m): # change to be a binned histogram
    com = shrinking_sphere(x, y, z, m)

    radii = np.sqrt( (x - com[0])**2 + (y - com[1])**2 + (z - com[2])**2 )

    idx = np.argsort(radii)
    radii_sort = radii[idx]
    m_sort = m[idx]

    return radii_sort, m_sort, np.cumsum(m_sort)

In [12]:
#changed to use a binned model sacrificing presision for speed
def find_mass_profile(x, y, z, m, com=None, bins=50, rmax=None):
    if com is None:
        com = shrinking_sphere(x, y, z, m)

    radii = np.sqrt( (x - com[0])**2 + (y - com[1])**2 + (z - com[2])**2 )
    
    if rmax is None:
        rmax = radii.max()

    log_edges = np.linspace(np.log10(radii.min()), np.log10(rmax), bins)
    mass_per_bin, _ = np.histogram(np.log10(radii), bins=log_edges, weights=m)
    rad_edges = 10**log_edges

    return rad_edges[:-1], mass_per_bin, np.cumsum(mass_per_bin)
    


def find_mass_profiles(run_data, bins=50, rmax=None):
    mass_profiles = {time : find_mass_profile(df['X'], df['Y'], df['Z'], df['M'], bins=bins, rmax=rmax)
                                                      for time,df in run_data.items()}
    return mass_profiles

def find_mass_profiles_experiment(runs_data, bins=50, rmax=None):
    mass_profiles_by_run = {run : find_mass_profiles(data, bins=bins, rmax=rmax) for run, (data,_) in runs_data.items()}
    return mass_profiles_by_run


In [3]:
def find_bounding_radii(x, y, z, m, fractions):
    com = shrinking_sphere(x, y, z, m)

    radii = np.sqrt( (x - com[0])**2 + (y - com[1])**2 + (z - com[2])**2 )

    

    idx = np.argsort(radii)
    radii_sort = radii[idx]
    m_sort = m[idx]
    
    m_cumul = np.cumsum(m_sort)

    m_fractions = m_cumul[-1] * fractions
    
    frac_mass_radii = radii_sort[np.searchsorted(m_cumul, m_fractions)]
    return frac_mass_radii

def find_bounding_radii_mp(mass_profile, fractions):
    radii_sort, m_sort, m_cumul = mass_profile

    m_fractions = fractions * m_cumul[-1]
    return radii_sort[np.searchsorted(m_cumul, m_fractions)]

def find_all_bounding_radiis(mass_profiles_by_run, fractions):
    return {
        run: pd.DataFrame(
            [[time, *find_bounding_radii_mp(mass_profile, fractions)]
             for time, mass_profile in mass_profiles_by_time.items()],
            columns=['time', *fractions])
        for run, mass_profiles_by_time in mass_profiles_by_run.items()}


def store_all_bounding_radiis(path, bounding_df_by_run):
    with pd.HDFStore(Path(path), mode='w') as store:
        for run, df in bounding_df_by_run.items():
            store[str(run)] = df

def read_all_bounding_radiis(path):
    with pd.HDFStore('bounding_radii_w0.h5', mode='r') as store:
        return {run: store[run] for run in store.keys()}


In [14]:
def plot_mass_profile(fig, ax, mass_profiles, 
                      xlbl=r'$r$ / $pc$', ylbl=r'$M(<r)$ / $M_\odot$', title='',
                      legend=False, marker=''
                      ):
    for time, (r_sort, _, m_sums) in mass_profiles.items():
        ax.plot(r_sort, m_sums, label=f'{time:.1f}Myr', marker=marker)
        
    ax.set_xscale('log')
    #plt.yscale('log')
    
    if legend: ax.legend()
    ax.set_xlabel(xlbl)
    ax.set_ylabel(ylbl)
    ax.set_title(title)

def plot_mass_profiles(fig, axes, runs_mass_profiles, titlehead='', legend=True, marker=''):
    for ax, (run, mass_profiles) in zip(axes.flat, runs_mass_profiles.items(), marker=''):
        plot_mass_profile(fig, ax, mass_profiles, title=rf'{titlehead}{run}', legend=legend,
                         marker=marker)
        